# ¿Para que esta esta Notebook?
Me he descarga un dataset de Kaggle con ventas de cafeterias, el dataset es como si hubiesen comprimido una base de datos en una sola tabla

Esta notebook esta hecha para poder dividir bien los datos, crear tablas nuevas y columnas nuevas, para así crear una base de datos mas "real"

Primero voy a editar el dataset original y crearé querys para MYSqL para que se guarden solas en la carpeta

Solo habria que ejecutar python para crear las queries e ir a workbench a ejecutarlas

In [1]:
import pandas as pd

In [ ]:
# Orden de creacion: Categories - Products - Stores - Employees - Sales
create_DB_and_categories = """
CREATE DATABASE IF NOT EXISTS coffeShopDB;
USE coffeShopDB;

CREATE TABLE IF NOT EXISTS categories (
    category_id INT AUTO_INCREMENT PRIMARY KEY,
    category_name varchar(100) NOT NULL
);
"""
create_products = """
CREATE TABLE IF NOT EXISTS products (
    product_id INT AUTO_INCREMENT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    product_type VARCHAR(100) NOT NULL,
    unti_price DECIMAL(10,2) NOT NULL,
    category_id INT NOT NULL,
    
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
    ON DELETE NO ACTION ON UPDATE CASCADE
);
"""
create_stores = """
CREATE TABLE IF NOT EXISTS stores (
    store_id INT AUTO_INCREMENT PRIMARY KEY,
    store_location VARCHAR(100) NOT NULL,
    store_capacity INT NOT NULL,
    store_status BOOLEAN NOT NULL
);
"""
create_employees = """
CREATE TABLE IF NOT EXISTS employees (
    employee_id INT AUTO_INCREMENT PRIMARY KEY,
    employee_name VARCHAR(100) NOT NULL,
    employee_hiring_date DATE NOT NULL,
    employee_position VARCHAR(80) NOT NULL,
    employee_dni varchar(10) NOT NULL UNIQUE,
    store_id INT NOT NULL,
    
    FOREIGN KEY (store_id) REFERENCES stores(store_id)
    ON DELETE NO ACTION ON UPDATE CASCADE
);
"""
create_sales = """
CREATE TABLE IF NOT EXISTS sales (
    sale_id INT AUTO_INCREMENT PRIMARY KEY,
    sale_date DATE NOT NULL,
    sale_quantity INT NOT NULL,
    sale_time TIME NOT NULL,
    sale_total DECIMAL(10,2) NOT NULL,
    store_id INT NOT NULL,
    product_id INT NOT NULL,
    employee_id INT NOT NULL,
    
    FOREIGN KEY (store_id) REFERENCES stores(store_id)
    ON DELETE NO ACTION ON UPDATE CASCADE,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
    ON DELETE NO ACTION ON UPDATE CASCADE,
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
    ON DELETE NO ACTION ON UPDATE CASCADE
);
"""
with open("../Queries/schema.sql", "w") as f:
    f.write(create_DB_and_categories)
    f.write(create_products)
    f.write(create_stores)
    f.write(create_employees)
    f.write(create_sales)

### Con el esquema acabado ahora falta rellenar los datos con todo lo del dataset

In [39]:
df = pd.read_csv("../Data/Coffee_Shop_Sales.csv",sep=";")

In [30]:
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_type,product_name,category_name
0,1,01/01/2023,7:06:11,2,5,Lower Manhattan,32,3,Gourmet brewed coffee,Ethiopia Rg,Coffee
1,2,01/01/2023,7:08:56,2,5,Lower Manhattan,57,"3,1",Brewed Chai tea,Spicy Eye Opener Chai Lg,Tea
2,3,01/01/2023,7:14:04,2,5,Lower Manhattan,59,"4,5",Hot chocolate,Dark chocolate Lg,Drinking Chocolate
3,4,01/01/2023,7:20:24,1,5,Lower Manhattan,22,2,Drip coffee,Our Old Time Diner Blend Sm,Coffee
4,5,01/01/2023,7:22:41,2,5,Lower Manhattan,57,"3,1",Brewed Chai tea,Spicy Eye Opener Chai Lg,Tea


In [ ]:
# Insercion de datos en la tabla stores

# el unico dato que recojo del dataset es store_location y creo que lo voy a cambiar por ciudades de España
# EN SQL HACER QUERY PARA ABRIR LA TIENDA DE NOJA Y METER EMPLEADOS NUEVOS
print(df["store_location"].unique()) # ['Lower Manhattan', 'Hell's Kitchen', 'Astoria'] originales

df["store_location"] = df["store_location"].replace({
    "Lower Manhattan": "Estepona",
    "Hell's Kitchen": "Torremolinos",
    "Astoria": "Noja"
})


insert_stores = """
USE coffeShopDB;

INSERT INTO stores (store_id,store_location, store_capacity, store_status) VALUES
(5, 'Estepona', 50, TRUE),
(8, 'Torremolinos', 30, TRUE),
(3, 'Noja', 60, FALSE);
"""

# for row in df.itertuples(index=False):    HACER ESTO PERO PARA ESCRIBIR LA QUERY
#     cursor.execute("""
#         INSERT INTO products
#         (product_name, product_type)
#         VALUES (%s, %s)
#     """, (row.product_name, row.product_type))

with open("../Queries/data.sql", "w") as f:
    f.write(insert_stores)

print(df["store_location"].unique())

<StringArray>
['Estepona', 'Torremolinos', 'Noja']
Length: 3, dtype: str
<StringArray>
['Estepona', 'Torremolinos', 'Noja']
Length: 3, dtype: str


In [ ]:
# Insercion de datos en la tabla categories

# en este caso solo nos importa el nombre
print(df["category_name"].unique()) # [Coffee, Tea, Drinking Chocolate, Bakery, Flavours, Loose Tea, Coffee beans, Packaged Chocolate, Branded]

insert_categories = """
INSERT INTO categories (category_name) VALUES
('Coffee'),
('Tea'),
('Drinking Chocolate'),
('Bakery'),
('Flavours'),
('Loose Tea'),
('Coffee beans'),
('Packaged Chocolate'),
('Branded');
"""

with open("../Queries/data.sql", "a") as f:
    f.write(insert_categories)

<StringArray>
[            'Coffee',                'Tea', 'Drinking Chocolate',
             'Bakery',           'Flavours',          'Loose Tea',
       'Coffee beans', 'Packaged Chocolate',            'Branded']
Length: 9, dtype: str


In [45]:
# Insercion de datos en la tabla employees

# como esta tabla se ha creado desde 0, no se usará nada del datset obviamente
# HACER UNA QUERY PARA CREAR UNA COLUMNA NUEVA QUE SEA PARA APELLIDOS

insert_employees = """
INSERT INTO employees (employee_name, employee_hiring_date, employee_position, employee_dni, store_id) VALUES
('Valeria Sánchez', '2022-03-15', 'Cashier', '12345678A', 5),
('Jorge Moreno', '2021-11-08', 'Supervisor', '23456789B', 5),
('Dario Fernández', '2023-01-20', 'Barista', '34567890C', 5),
('Ana Torres', '2021-06-10', 'Barista', '45678901D', 5),
('Marina Jiménez', '2024-02-05', 'Barista', '56789012E', 5),
('David Ruiz', '2022-09-12', 'Barista', '67890123F', 8),
('Lara Lépez', '2021-07-01', 'Supervisor', '78901234G', 8),
('Javier Martín', '2023-05-18', 'Cashier', '89012345I', 8);
"""

with open("../Queries/data.sql", "a",encoding="utf-8") as f:
    f.write(insert_employees)

select emp.employee_name, st.store_location
from employees emp join stores st on emp.store_id = st.store_id

In [ ]:
# hay que ordenar  el producto por el id que tiene, tipo nombre y precio por unidad
# no puedo crear ids nuevos pq ya estan asociados a ventas
# he tenido que guardarlo en cvs y todo para poder hacerlo manualmente
products = (
    df[["product_id", "product_name", "product_type", "unit_price"]]
    .drop_duplicates()
    .sort_values("product_id")
)

print(products)

      product_id               product_name       product_type unit_price
3451           1        Brazilian - Organic      Organic Beans         18
4494           2   Our Old Time Diner Blend  House blend Beans         18
3968           3             Espresso Roast     Espresso Beans      14,75
3554           4       Primo Espresso Roast     Espresso Beans      20,45
4328           5     Columbian Medium Roast      Gourmet Beans         15
...          ...                        ...                ...        ...
4033          83  I Need My Bean! Latte cup         Housewares         14
7711          83  I Need My Bean! Latte cup         Housewares         23
3352          84            Chocolate syrup      Regular syrup        0,8
14            87       Ouro Brasileiro shot   Barista Espresso          3
8242          87       Ouro Brasileiro shot   Barista Espresso        2,1

[98 rows x 4 columns]


In [ ]:
    # product_id INT AUTO_INCREMENT PRIMARY KEY,
    # product_name VARCHAR(100) NOT NULL,
    # product_type VARCHAR(100) NOT NULL,
    # product_size INT NOT NULL, -- se rellena en python ya que es inventado (1, 2, 3)
    # unti_price DECIMAL(10,2) NOT NULL,
    # category_id INT NOT NULL,
# 1	Coffee
# 2	Tea
# 3	Drinking Chocolate
# 4	Bakery
# 5	Flavours
# 6	Loose Tea
# 7	Coffee beans
# 8	Packaged Chocolate
# 9	Branded

insert_products = """
INSERT INTO products (product_id, product_name, product_type, unit_price, category_id) VALUES
(1, 'Brazilian - Organic', 'Organic Beans', 18),
(2, 'Our Old Time Diner Blend', 'House blend Beans', 18),
(3, 'Espresso Roast', 'Espresso Beans', 14.75),
(4, 'Primo Espresso Roast', 'Espresso Beans', 20.45),
(5, 'Columbian Medium Roast', 'Gourmet Beans', 15),
(6, 'Ethiopia', 'Gourmet Beans', 21),
(7, 'Jamacian Coffee River', 'Premium Beans', 19.75),
(8, 'Civet Cat', 'Premium Beans', 45),
(10, 'Guatemalan Sustainably Grown', 'Green beans', 10),
(11, 'Lemon Grass', 'Herbal tea', 8.95),
(12, 'Peppermint', 'Herbal tea', 8.95),
(13, 'English Breakfast', 'Black tea', 8.95),
(14, 'Earl Grey', 'Black tea', 8.95),
(15, 'Serenity Green Tea', 'Green tea', 9.25),
(16, 'Traditional Blend Chai', 'Chai tea', 8.95),
(17, 'Morning Sunrise Chai', 'Chai tea', 9.50),
(18, 'Spicy Eye Opener Chai', 'Chai tea', 10.95),
(19, 'Dark chocolate', 'Drinking Chocolate', 6.40),
(20, 'Sustainably Grown Organic', 'Organic Chocolate', 7.60),
(21, 'Chili Mayan', 'Drinking Chocolate', 13.33),
(22, 'Our Old Time Diner Blend Sm', 'Drip coffee', 2.00),
(23, 'Our Old Time Diner Blend Rg', 'Drip coffee', 2.50),
(24, 'Our Old Time Diner Blend Lg', 'Drip coffee', 3.00),
(25, 'Brazilian Sm', 'Organic brewed coffee', 2.20),
(26, 'Brazilian Rg', 'Organic brewed coffee', 3.00),
(27, 'Brazilian Lg', 'Organic brewed coffee', 3.50),
(28, 'Columbian Medium Roast Sm', 'Gourmet brewed coffee', 2.00),
(29, 'Columbian Medium Roast Rg', 'Gourmet brewed coffee', 2.50),
(30, 'Columbian Medium Roast Lg', 'Gourmet brewed coffee', 3.00),
(31, 'Ethiopia Sm', 'Gourmet brewed coffee', 2.20),
(32, 'Ethiopia Rg', 'Gourmet brewed coffee', 3.00),
(33, 'Ethiopia Lg', 'Gourmet brewed coffee', 3.50),
(34, 'Jamaican Coffee River Sm', 'Premium brewed coffee', 2.45),
(35, 'Jamaican Coffee River Rg', 'Premium brewed coffee', 3.10),
(36, 'Jamaican Coffee River Lg', 'Premium brewed coffee', 3.75),
(37, 'Espresso shot', 'Barista Espresso', 3.00),
(38, 'Latte', 'Barista Espresso', 3.75),
(39, 'Latte Rg', 'Barista Espresso', 4.25),
(40, 'Cappuccino', 'Barista Espresso', 3.75),
(41, 'Cappuccino Lg', 'Barista Espresso', 4.25),
(42, 'Lemon Grass Rg', 'Brewed herbal tea', 2.50),
(43, 'Lemon Grass Lg', 'Brewed herbal tea', 3.00),
(44, 'Peppermint Rg', 'Brewed herbal tea', 2.50),
(45, 'Peppermint Lg', 'Brewed herbal tea', 3.00),
(46, 'Serenity Green Tea Rg', 'Brewed Green tea', 2.50),
(47, 'Serenity Green Tea Lg', 'Brewed Green tea', 3.00),
(48, 'English Breakfast Rg', 'Brewed Black tea', 2.50),
(49, 'English Breakfast Lg', 'Brewed Black tea', 3.00),
(50, 'Earl Grey Rg', 'Brewed Black tea', 2.50),
(51, 'Earl Grey Lg', 'Brewed Black tea', 3.00),
(52, 'Traditional Blend Chai Rg', 'Brewed Chai tea', 2.50),
(53, 'Traditional Blend Chai Lg', 'Brewed Chai tea', 3.00),
(54, 'Morning Sunrise Chai Rg', 'Brewed Chai tea', 2.50),
(55, 'Morning Sunrise Chai Lg', 'Brewed Chai tea', 4.00),
(56, 'Spicy Eye Opener Chai Rg', 'Brewed Chai tea', 2.55),
(57, 'Spicy Eye Opener Chai Lg', 'Brewed Chai tea', 3.10),
(58, 'Dark chocolate Rg', 'Hot chocolate', 3.50),
(59, 'Dark chocolate Lg', 'Hot chocolate', 4.50),
(60, 'Sustainably Grown Organic Rg', 'Hot chocolate', 3.75),
(61, 'Sustainably Grown Organic Lg', 'Hot chocolate', 4.75),
(63, 'Carmel syrup', 'Regular syrup', 0.80),
(64, 'Hazelnut syrup', 'Regular syrup', 0.80),
(65, 'Sugar Free Vanilla syrup', 'Sugar free syrup', 0.80),
(69, 'Hazelnut Biscotti', 'Biscotti', 4.06),
(70, 'Cranberry Scone', 'Scone', 4.06),
(71, 'Chocolate Croissant', 'Pastry', 4.69),
(72, 'Ginger Scone', 'Scone', 4.06),
(73, 'Almond Croissant', 'Pastry', 4.69),
(74, 'Ginger Biscotti', 'Biscotti', 4.38),
(75, 'Croissant', 'Pastry', 4.38),
(76, 'Chocolate Chip Biscotti', 'Biscotti', 4.38),
(77, 'Oatmeal Scone', 'Scone', 3.00),
(78, 'Scottish Cream Scone', 'Scone', 5.63),
(79, 'Jumbo Savory Scone', 'Scone', 4.69),
(81, 'I Need My Bean! T-shirt', 'Clothing', 28.00),
(82, 'I Need My Bean! Diner mug', 'Housewares', 23.00),
(83, 'I Need My Bean! Latte cup', 'Housewares', 23.00),
(84, 'Chocolate syrup', 'Regular syrup', 0.80),
(87, 'Ouro Brasileiro shot', 'Barista Espresso', 3.00);
"""

with open("../Queries/data.sql", "a",encoding="utf-8") as f:
    f.write(insert_employees)
    
